## Organize reachable leisure pois

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
# Load libs
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm
import time
from lib import helpers as helpers
import numpy as np

## 1. Load w-k results and process
Reachable POIs, and the travel times from workplace to them.

- Situation 1: Real time budget
- Situation 2: 90 min uniform time budget
- Situation 3: 90 min one-way uniform time budget

In [3]:
df_com = pd.read_csv("dbs/data_p/commuter_time_budget.csv")
df_com.head()

,ID,time_budget,time_hw,tt_wkh_1,tt_wkh_2,tt_wkh_3
0,10_2978,184.0,25.0,134.0,40.0,50.0
1,10_2980,57.5,21.0,15.5,48.0,54.0
2,10_2981,91.0,43.5,4.0,3.0,31.5
3,10_2982,228.0,106.0,16.0,-122.0,-31.0
4,10_2984,39.0,20.0,-1.0,50.0,55.0


In [ ]:
def process_mode(mode):
    df_list = []
    candidate_individuals = dict()
    for situation in ['01', '02', '03']:
        candidate_individuals[situation] = []
        for tbin in ['15', '30', '45', '60', '75']:
            # Load the results
            print(f'Loading mode {mode}, situation {situation}, time bin {tbin}.')
            df = pd.read_csv(f'dbs/sp_accessibility/tt_wk_{mode}_17_{situation}_{tbin}.csv')
            df = pd.merge(df, df_com[['ID', 'tt_wkh_1', 'tt_wkh_2', 'tt_wkh_3']], left_on='from_id', right_on='ID', how='left')
            for v, nv in zip(['tt_wkh_1', 'tt_wkh_2', 'tt_wkh_3'], ['tt_kh_1', 'tt_kh_2', 'tt_kh_3']):
                df[nv] = df[v] - df['travel_time_p50']
            # Collect candidate POIs and individuals
            sit = int(situation)
            candidate_individuals[situation].extend(df[df[f'tt_kh_{sit}'] > 0]['from_id'].unique().tolist())
            candidate_individuals[situation] = list(set(candidate_individuals[situation]))
            temp = df[df[f'tt_kh_{sit}'] > 0][['ID', 'to_id', f'tt_kh_{sit}']].copy().rename(columns={f'tt_kh_{sit}': 'tt_kh'})
            temp['mode'] = mode
            temp['situation'] = situation
            df_list.append(temp)
    df_p = pd.concat(df_list, ignore_index=True)
    return df_p, candidate_individuals

In [7]:
# Revised situation = 2 (90 min budget)
def process_mode_r(mode):
    df_list = []
    candidate_individuals = []
    for tbin in ['15', '30', '45', '60', '75', '90']:
        # Load the results
        print(f'Loading mode {mode}, time bin {tbin}.')
        df = pd.read_csv(f'dbs/sp_accessibility_r/tt_wk_{mode}_17_{tbin}.csv')
        df = pd.merge(df, df_com[['ID', 'tt_wkh_2']], left_on='from_id', right_on='ID', how='left')
        for v, nv in zip(['tt_wkh_2'], ['tt_kh_2']):
            df[nv] = df[v] - df['travel_time_p50']
        # Collect candidate POIs and individuals
        candidate_individuals.extend(df[df[f'tt_kh_2'] > 0]['from_id'].unique().tolist())
        candidate_individuals = list(set(candidate_individuals))
        temp = df[df[f'tt_kh_2'] > 0][['ID', 'to_id', f'tt_kh_2']].copy().rename(columns={f'tt_kh_2': 'tt_kh'})
        temp['mode'] = mode
        df_list.append(temp)
    df_p = pd.concat(df_list, ignore_index=True)
    return df_p, candidate_individuals

## 2. Work-to-k results

In [5]:
df_p_1, candidate_individuals_1 = process_mode('pt')
for situation in ['01', '02', '03']:
    no_pois = df_p_1[df_p_1['situation'] == situation]['to_id'].nunique()
    print(f'Situation {situation}: {len(candidate_individuals_1[situation])} individuals, {no_pois} POIs.')

Loading mode pt, situation 01, time bin 15.
Loading mode pt, situation 01, time bin 30.
Loading mode pt, situation 01, time bin 45.
Loading mode pt, situation 01, time bin 60.
Loading mode pt, situation 01, time bin 75.
Loading mode pt, situation 02, time bin 15.
Loading mode pt, situation 02, time bin 30.
Loading mode pt, situation 02, time bin 45.
Loading mode pt, situation 02, time bin 60.
Loading mode pt, situation 02, time bin 75.
Loading mode pt, situation 03, time bin 15.
Loading mode pt, situation 03, time bin 30.
Loading mode pt, situation 03, time bin 45.
Loading mode pt, situation 03, time bin 60.
Loading mode pt, situation 03, time bin 75.
Situation 01: 891 individuals, 19297 POIs.
Situation 02: 1105 individuals, 19368 POIs.
Situation 03: 1989 individuals, 19641 POIs.


In [5]:
df_p_2, candidate_individuals_2 = process_mode('car')
for situation in ['01', '02', '03']:
    no_pois = df_p_2[df_p_2['situation'] == situation]['to_id'].nunique()
    print(f'Situation {situation}: {len(candidate_individuals_2[situation])} individuals, {no_pois} POIs.')

Loading mode car, situation 01, time bin 15.
Loading mode car, situation 01, time bin 30.
Loading mode car, situation 01, time bin 45.
Loading mode car, situation 01, time bin 60.
Loading mode car, situation 01, time bin 75.
Loading mode car, situation 02, time bin 15.
Loading mode car, situation 02, time bin 30.
Loading mode car, situation 02, time bin 45.
Loading mode car, situation 02, time bin 60.
Loading mode car, situation 02, time bin 75.
Loading mode car, situation 03, time bin 15.
Loading mode car, situation 03, time bin 30.
Loading mode car, situation 03, time bin 45.
Loading mode car, situation 03, time bin 60.
Loading mode car, situation 03, time bin 75.
Situation 01: 999 individuals, 20412 POIs.
Situation 02: 1169 individuals, 20411 POIs.
Situation 03: 2051 individuals, 20411 POIs.


### 2.1 Updated SPA (90 min situation 2)

In [8]:
df_p_1, candidate_individuals_1 = process_mode_r('pt')
no_pois = df_p_1['to_id'].nunique()
print(f'Situation 2: {len(candidate_individuals_1)} individuals, {no_pois} POIs.')

Loading mode pt, time bin 15.
Loading mode pt, time bin 30.
Loading mode pt, time bin 45.
Loading mode pt, time bin 60.
Loading mode pt, time bin 75.
Loading mode pt, time bin 90.
Situation 2: 1372 individuals, 42672 POIs.


In [9]:
df_p_2, candidate_individuals_2 = process_mode_r('car')
no_pois = df_p_2['to_id'].nunique()
print(f'Situation 2: {len(candidate_individuals_1)} individuals, {no_pois} POIs.')

Loading mode car, time bin 15.
Loading mode car, time bin 30.
Loading mode car, time bin 45.
Loading mode car, time bin 60.
Loading mode car, time bin 75.
Loading mode car, time bin 90.
Situation 2: 1372 individuals, 44580 POIs.


## 3. Data preparation for k-to-Home
### 3.1 Prepare destinations

In [10]:
df_work = pd.read_csv('dbs/data_p/commuter_trips.csv')
df_work = df_work[df_work['purpose_d'] == 'HOME'].copy()
df_work.drop_duplicates(subset=['ID'], inplace=True)
df_work = df_work[['ID', 'end_lon', 'end_lat']].rename(columns={'ID': 'id', 'end_lon': 'lon', 'end_lat': 'lat'})

In [8]:
mode = 'pt'
for situation in ['01', '02', '03']:
    df_work[df_work['id'].isin(candidate_individuals_1[situation])].to_csv(f'dbs/sp_accessibility/data/destinations_{mode}_{int(situation)}.csv', index=False)

In [9]:
mode = 'car'
for situation in ['01', '02', '03']:
    df_work[df_work['id'].isin(candidate_individuals_2[situation])].to_csv(f'dbs/sp_accessibility/data/destinations_{mode}_{int(situation)}.csv', index=False)

#### Updated

In [11]:
mode = 'pt'
df_work[df_work['id'].isin(candidate_individuals_1)].to_csv(f'dbs/sp_accessibility_r/data/destinations_{mode}.csv', index=False)
mode = 'car'
df_work[df_work['id'].isin(candidate_individuals_2)].to_csv(f'dbs/sp_accessibility_r/data/destinations_{mode}.csv', index=False)

### 3.2 Prepare origins

In [12]:
df2save = pd.read_csv("dbs/sp_accessibility_r/data/destinations.csv")
df2save.head()

,id,lon,lat
0,08f1fb648e64a1b2036a293412ffbe11,1.051928,47.807925
1,08f186c8714235a20313d85fc8b4a322,1.370320,47.496949
2,08f186c871509433037abfcd3b905ecd,1.372060,47.497726
3,08f186c871c85974032cd77c85874516,1.373229,47.502235
4,08f186c862543d8e0349c57a46276a02,1.403493,47.498089


In [13]:
df_p_2.head()

,ID,to_id,tt_kh,mode
0,10_2981,08f1fb08a3709d3603bf93e752d94154,1.0,car
1,10_3025,08f1fb4658560c510346e5f6d94b7c5b,1.0,car
2,10_3025,08f1fb4658560ba003611c53294b1dd1,1.0,car
3,10_3025,08f1fb4658189c10030ed2fd4f7a805c,1.0,car
4,10_3025,08f1fb465800039203f0f533572a2f42,1.0,car


In [9]:
# Define bins and labels (right-exclusive, so [0,15), [15,30), …)
bins   = [0, 15, 30, 45, 60, 75, 90.1]
labels = [15, 30, 45, 60, 75, 90]   # dictionary keys
mode = 'car'    # car, pt and change df_p_2 to df_p_1 for public transport
for situation in ['01', '02', '03']:
    temp = df_p_2[df_p_2['situation'] == situation].copy()
    temp['tt_kh'] = temp['tt_kh'].apply(lambda x: x if x <= 90 else 90)  # Cap at 90 minutes
    temp['group'] = pd.cut(temp['tt_kh'], bins=bins, labels=labels, right=False)
    # Build dictionary: {15: [...IDs...], 30: [...], …}
    group_dict = (
        temp.groupby('group')['to_id']
            .apply(lambda x: list(set(x)))
            .to_dict()
    )
    for lb in labels:
        df2save_lb = df2save[df2save['id'].isin(group_dict[lb])].copy()
        print("No. of POIs", len(df2save_lb))
        if len(df2save_lb) > 0:
            if (len(df2save_lb) > 10000) & (mode == 'car'):
                # Split into 8 batches
                batches = np.array_split(df2save_lb, 8)
                for i, batch in enumerate(batches, start=1):
                    out_path = f'dbs/sp_accessibility/data/origins_{mode}_{int(situation)}_{lb}_part{i}.csv'
                    batch[['id', 'lon', 'lat']].to_csv(out_path, index=False)
                    print(f"Saved batch {i} with {len(batch)} rows → {out_path}")
            else:
                # Save normally
                out_path = f'dbs/sp_accessibility/data/origins_{mode}_{int(situation)}_{lb}.csv'
                df2save_lb[['id', 'lon', 'lat']].to_csv(out_path, index=False)
                print(f"Saved single file with {len(df2save_lb)} rows → {out_path}")

C:\Users\yuanlia\AppData\Local\Temp\ipykernel_19248\1092724506.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp.groupby('group')['to_id']


No. of POIs 20398
Saved batch 1 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_15_part1.csv
Saved batch 2 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_15_part2.csv
Saved batch 3 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_15_part3.csv
Saved batch 4 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_15_part4.csv
Saved batch 5 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_15_part5.csv
Saved batch 6 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_15_part6.csv
Saved batch 7 with 2549 rows → dbs/sp_accessibility/data/origins_car_1_15_part7.csv
Saved batch 8 with 2549 rows → dbs/sp_accessibility/data/origins_car_1_15_part8.csv


c:\ProgramData\anaconda3\envs\mobi\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


No. of POIs 20404
Saved batch 1 with 2551 rows → dbs/sp_accessibility/data/origins_car_1_30_part1.csv
Saved batch 2 with 2551 rows → dbs/sp_accessibility/data/origins_car_1_30_part2.csv
Saved batch 3 with 2551 rows → dbs/sp_accessibility/data/origins_car_1_30_part3.csv
Saved batch 4 with 2551 rows → dbs/sp_accessibility/data/origins_car_1_30_part4.csv
Saved batch 5 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_30_part5.csv
Saved batch 6 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_30_part6.csv
Saved batch 7 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_30_part7.csv
Saved batch 8 with 2550 rows → dbs/sp_accessibility/data/origins_car_1_30_part8.csv


c:\ProgramData\anaconda3\envs\mobi\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


...

c:\ProgramData\anaconda3\envs\mobi\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Saved batch 1 with 2542 rows → dbs/sp_accessibility/data/origins_car_3_45_part1.csv
Saved batch 2 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part2.csv
Saved batch 3 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part3.csv
Saved batch 4 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part4.csv
Saved batch 5 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part5.csv
Saved batch 6 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part6.csv
Saved batch 7 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part7.csv
Saved batch 8 with 2541 rows → dbs/sp_accessibility/data/origins_car_3_45_part8.csv
No. of POIs 19987


c:\ProgramData\anaconda3\envs\mobi\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Saved batch 1 with 2499 rows → dbs/sp_accessibility/data/origins_car_3_60_part1.csv
Saved batch 2 with 2499 rows → dbs/sp_accessibility/data/origins_car_3_60_part2.csv
Saved batch 3 with 2499 rows → dbs/sp_accessibility/data/origins_car_3_60_part3.csv
Saved batch 4 with 2498 rows → dbs/sp_accessibility/data/origins_car_3_60_part4.csv
Saved batch 5 with 2498 rows → dbs/sp_accessibility/data/origins_car_3_60_part5.csv
Saved batch 6 with 2498 rows → dbs/sp_accessibility/data/origins_car_3_60_part6.csv
Saved batch 7 with 2498 rows → dbs/sp_accessibility/data/origins_car_3_60_part7.csv
Saved batch 8 with 2498 rows → dbs/sp_accessibility/data/origins_car_3_60_part8.csv
No. of POIs 13746
Saved batch 1 with 1719 rows → dbs/sp_accessibility/data/origins_car_3_75_part1.csv
Saved batch 2 with 1719 rows → dbs/sp_accessibility/data/origins_car_3_75_part2.csv
Saved batch 3 with 1718 rows → dbs/sp_accessibility/data/origins_car_3_75_part3.csv
Saved batch 4 with 1718 rows → dbs/sp_accessibility/data/o

c:\ProgramData\anaconda3\envs\mobi\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


### 3.3 Updated SPA (90 min situation 2)

In [16]:
# Updated situation 2 (90 min travel time budget)
# Define bins and labels (right-exclusive, so [0,15), [15,30), …)
bins   = [0, 15, 30, 45, 60, 75, 90.1]
labels = [15, 30, 45, 60, 75, 90]   # dictionary keys
mode = 'pt'    # car, pt and change df_p_2 to df_p_1 for public transport
temp = df_p_1.copy()
temp['tt_kh'] = temp['tt_kh'].apply(lambda x: x if x <= 90 else 90)  # Cap at 90 minutes
temp['group'] = pd.cut(temp['tt_kh'], bins=bins, labels=labels, right=False)
# Build dictionary: {15: [...IDs...], 30: [...], …}
group_dict = (
    temp.groupby('group')['to_id']
        .apply(lambda x: list(set(x)))
        .to_dict()
)
for lb in labels:
    df2save_lb = df2save[df2save['id'].isin(group_dict[lb])].copy()
    print("No. of POIs", len(df2save_lb))
    if len(df2save_lb) > 0:
        if (len(df2save_lb) > 10000) & (mode == 'car'):
            # Split into 8 batches
            batches = np.array_split(df2save_lb, 8)
            for i, batch in enumerate(batches, start=1):
                out_path = f'dbs/sp_accessibility_r/data/origins_{mode}_{lb}_part{i}.csv'
                batch[['id', 'lon', 'lat']].to_csv(out_path, index=False)
                print(f"Saved batch {i} with {len(batch)} rows → {out_path}")
        else:
            # Save normally
            out_path = f'dbs/sp_accessibility_r/data/origins_{mode}_{lb}.csv'
            df2save_lb[['id', 'lon', 'lat']].to_csv(out_path, index=False)
            print(f"Saved single file with {len(df2save_lb)} rows → {out_path}")

C:\Users\yuanlia\AppData\Local\Temp\ipykernel_20608\3398924105.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  temp.groupby('group')['to_id']


No. of POIs 42291
Saved single file with 42291 rows → dbs/sp_accessibility_r/data/origins_pt_15.csv
No. of POIs 41167
Saved single file with 41167 rows → dbs/sp_accessibility_r/data/origins_pt_30.csv
No. of POIs 38745
Saved single file with 38745 rows → dbs/sp_accessibility_r/data/origins_pt_45.csv
No. of POIs 32796
Saved single file with 32796 rows → dbs/sp_accessibility_r/data/origins_pt_60.csv
No. of POIs 15926
Saved single file with 15926 rows → dbs/sp_accessibility_r/data/origins_pt_75.csv
No. of POIs 431
Saved single file with 431 rows → dbs/sp_accessibility_r/data/origins_pt_90.csv
